# StudyFlow Full Python, Statistics, SQL and Product Analytics

Полный readable notebook: Python/Pandas, продуктовые метрики, статистика, A/B и SQL.


In [ ]:
from pathlib import Path
import math
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

users = pd.read_csv(RAW / 'users.csv', parse_dates=['signup_date', 'assigned_at'])
events = pd.read_csv(RAW / 'events.csv', parse_dates=['event_time'])
payments = pd.read_csv(RAW / 'payments.csv', parse_dates=['payment_date'])
support = pd.read_csv(RAW / 'support_tickets.csv', parse_dates=['created_at'])
active_users = users[users['is_test_account'] == False].copy()
users.shape, active_users.shape


## 1. Data Quality


In [ ]:
missing = users.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
quality_report = pd.DataFrame({'missing_rows': missing, 'missing_share': missing / len(users)})
print('user_id unique:', users['user_id'].is_unique)
quality_report


## 2. Product KPI


In [ ]:
kpi = pd.Series({
    'users': active_users['user_id'].nunique(),
    'activation_rate_7d': active_users['activated_7d'].mean(),
    'trial_start_rate_7d': active_users['trial_started'].mean(),
    'paid_conversion_14d': active_users['paid_14d'].mean(),
    'arpu_30d': active_users['revenue_30d'].mean(),
    'arppu_30d': active_users.loc[active_users['paid_14d'], 'revenue_30d'].mean(),
    'retention_30d': active_users['retained_30d'].mean(),
    'retention_60d': active_users['retained_60d'].mean(),
    'refund_rate_payers_30d': active_users.loc[active_users['paid_14d'], 'refund_30d'].mean(),
    'payment_failed_rate': active_users['payment_failed'].mean(),
    'avg_nps': active_users['nps_score'].mean(),
})
kpi


## 3. Funnel


In [ ]:
funnel = pd.DataFrame([
    {'step': 'signup', 'users': active_users['user_id'].nunique()},
    {'step': 'session_start', 'users': active_users.loc[active_users['sessions_7d'] > 0, 'user_id'].nunique()},
    {'step': 'lesson_started', 'users': active_users.loc[active_users['lessons_started_7d'] > 0, 'user_id'].nunique()},
    {'step': 'activated_7d', 'users': active_users.loc[active_users['activated_7d'], 'user_id'].nunique()},
    {'step': 'paywall_seen', 'users': active_users.loc[active_users['paywall_seen'], 'user_id'].nunique()},
    {'step': 'trial_started', 'users': active_users.loc[active_users['trial_started'], 'user_id'].nunique()},
    {'step': 'paid_14d', 'users': active_users.loc[active_users['paid_14d'], 'user_id'].nunique()},
    {'step': 'retained_30d', 'users': active_users.loc[active_users['retained_30d'], 'user_id'].nunique()},
])
funnel['from_signup_rate'] = funnel['users'] / funnel.loc[0, 'users']
funnel['step_to_step_rate'] = funnel['users'] / funnel['users'].shift(1)
funnel


## 4. Channel Unit Economics


In [ ]:
channel = active_users.groupby('channel').agg(
    users=('user_id', 'nunique'),
    activated=('activated_7d', 'sum'),
    payers=('paid_14d', 'sum'),
    revenue_30d=('revenue_30d', 'sum'),
    spend=('marketing_spend_user', 'sum'),
    retained_30d=('retained_30d', 'sum'),
    refunds=('refund_30d', 'sum'),
    payment_failed=('payment_failed', 'sum'),
).reset_index()
channel['activation_rate'] = channel['activated'] / channel['users']
channel['paid_conversion'] = channel['payers'] / channel['users']
channel['arpu'] = channel['revenue_30d'] / channel['users']
channel['cac'] = channel['spend'] / channel['payers'].replace(0, np.nan)
channel['roas'] = channel['revenue_30d'] / channel['spend'].replace(0, np.nan)
channel['profit_proxy'] = channel['revenue_30d'] - channel['spend']
channel['retention_30d'] = channel['retained_30d'] / channel['users']
channel.sort_values('profit_proxy', ascending=False)


## 5. Cohorts and Segments


In [ ]:
cohort = active_users.groupby('cohort_month').agg(
    users=('user_id', 'nunique'),
    activation_rate=('activated_7d', 'mean'),
    paid_conversion=('paid_14d', 'mean'),
    retention_30d=('retained_30d', 'mean'),
    retention_60d=('retained_60d', 'mean'),
    arpu=('revenue_30d', 'mean'),
).reset_index()
segment = active_users.groupby('user_segment').agg(
    users=('user_id', 'nunique'),
    activation_rate=('activated_7d', 'mean'),
    paid_conversion=('paid_14d', 'mean'),
    arpu=('revenue_30d', 'mean'),
    retention_30d=('retained_30d', 'mean'),
    avg_lessons_completed=('lessons_completed_7d', 'mean'),
).sort_values('arpu', ascending=False)
cohort, segment


## 6. Statistics: CI, t-test, chi-square, correlation


In [ ]:
paid_cr = active_users['paid_14d'].mean()
paid_se = math.sqrt(paid_cr * (1 - paid_cr) / len(active_users))
paid_ci = (paid_cr - 1.96 * paid_se, paid_cr + 1.96 * paid_se)

arpu = active_users['revenue_30d'].mean()
arpu_se = active_users['revenue_30d'].std(ddof=1) / math.sqrt(len(active_users))
arpu_ci = (arpu - 1.96 * arpu_se, arpu + 1.96 * arpu_se)

t_stat, t_p = stats.ttest_ind(
    active_users.loc[active_users['activated_7d'], 'study_minutes_7d'],
    active_users.loc[~active_users['activated_7d'], 'study_minutes_7d'],
    equal_var=False,
)
chi2, chi_p, _, _ = stats.chi2_contingency(pd.crosstab(active_users['device'], active_users['payment_failed']))
corr = active_users['lessons_completed_7d'].corr(active_users['quiz_score_after'])

pd.DataFrame([
    {'metric': 'paid_conversion_ci', 'value': paid_cr, 'ci_low': paid_ci[0], 'ci_high': paid_ci[1], 'p_value': np.nan},
    {'metric': 'arpu_ci', 'value': arpu, 'ci_low': arpu_ci[0], 'ci_high': arpu_ci[1], 'p_value': np.nan},
    {'metric': 'study_minutes_ttest', 'value': t_stat, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': t_p},
    {'metric': 'device_payment_failed_chi2', 'value': chi2, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': chi_p},
    {'metric': 'lessons_quiz_corr', 'value': corr, 'ci_low': np.nan, 'ci_high': np.nan, 'p_value': np.nan},
])


## 7. A/B Test: Smart Onboarding


In [ ]:
ab_summary = active_users.groupby('experiment_group').agg(
    users=('user_id', 'nunique'),
    activated=('activated_7d', 'sum'),
    payers=('paid_14d', 'sum'),
    arpu=('revenue_30d', 'mean'),
    retained_30d=('retained_30d', 'mean'),
    payment_failed_rate=('payment_failed', 'mean'),
    refund_rate=('refund_30d', 'mean'),
    avg_support_tickets=('support_tickets_30d', 'mean'),
    avg_nps=('nps_score', 'mean'),
)
ab_summary['activation_rate'] = ab_summary['activated'] / ab_summary['users']
ab_summary['paid_conversion'] = ab_summary['payers'] / ab_summary['users']
rows = []
for metric_name, success_col, rate_col in [('activation_rate_7d', 'activated', 'activation_rate'), ('paid_conversion_14d', 'payers', 'paid_conversion')]:
    n1 = ab_summary.loc['control', 'users']
    n2 = ab_summary.loc['smart_onboarding', 'users']
    x1 = ab_summary.loc['control', success_col]
    x2 = ab_summary.loc['smart_onboarding', success_col]
    p1 = ab_summary.loc['control', rate_col]
    p2 = ab_summary.loc['smart_onboarding', rate_col]
    pooled = (x1 + x2) / (n1 + n2)
    se = math.sqrt(pooled * (1 - pooled) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    rows.append({'metric': metric_name, 'control': p1, 'smart_onboarding': p2, 'uplift_pp': (p2-p1)*100, 'z_stat': z, 'p_value': p_value})
pd.DataFrame(rows), ab_summary


## 8. SQL from Python


In [ ]:
conn = sqlite3.connect(PROCESSED / 'studyflow.sqlite')
sql = '''
WITH channel_base AS (
    SELECT channel,
           COUNT(*) AS users,
           SUM(CASE WHEN paid_14d THEN 1 ELSE 0 END) AS payers,
           SUM(revenue_30d) AS revenue_30d,
           SUM(marketing_spend_user) AS spend
    FROM users
    WHERE is_test_account = 0
    GROUP BY channel
)
SELECT channel, users, payers, revenue_30d, spend,
       1.0 * payers / users AS paid_conversion,
       1.0 * spend / NULLIF(payers, 0) AS cac,
       1.0 * revenue_30d / NULLIF(spend, 0) AS roas,
       revenue_30d - spend AS profit_proxy
FROM channel_base
ORDER BY profit_proxy DESC;
'''
sql_channel = pd.read_sql_query(sql, conn)
conn.close()
sql_channel


## 9. Final Business Findings


In [ ]:
findings = [
    'Smart onboarding improves activation and paid conversion; rollout is reasonable with guardrail monitoring.',
    'Organic, email and referral are stronger economically than paid channels by profit proxy.',
    'Mobile web requires payment failure diagnostics.',
    'Growth should optimize activation -> trial -> paid -> retained, not one isolated metric.',
]
findings
